# 🤖 SAP SD Order Validation Agent — MCP Protocol
**Autor:** Fabian Andrés Acevedo R. | C_S4CPB_2602 · C_LCNC_2601

**Stack:** FastMCP (Anthropic) · SAP S/4HANA Cloud · Clean Core · BTP

---
### Arquitectura
```
Usuario → MCP Client → SAP SD MCP Server → Claude Agent
                           ↓
              validate_material_number()
              check_pricing_conditions()
              verify_customer_credit_limit()
              create_sales_order()
                           ↓
                   S/4HANA Cloud API  ← Clean Core ✅
```
---
**Ejecuta las celdas en orden. Tiempo estimado: 3 minutos.**

## CELDA 1 — Instalación de dependencias

In [ ]:
# Instala las librerías necesarias
# mcp: el protocolo de Anthropic
# anthropic: el cliente para llamar a Claude
# nest_asyncio: necesario para correr async en Colab

!pip install -q mcp anthropic nest_asyncio
print('✅ Dependencias instaladas')

## CELDA 2 — API Key de Anthropic
> Obtén tu API key en: https://console.anthropic.com/

In [ ]:
import os
from google.colab import userdata

try:
    README.md
except Exception:
   README.md

## CELDA 3 — Servidor MCP SAP SD (el corazón del proyecto)

In [ ]:
# Guarda el servidor MCP en disco para que el cliente pueda lanzarlo como subproceso

sap_sd_server_code = '''
from mcp.server.fastmcp import FastMCP
from pydantic import Field
from mcp.server.fastmcp.prompts import base

mcp = FastMCP("SAP_SD_Validation_Agent", log_level="ERROR")

# ── DATOS MOCK ─────────────────────────────────────────────
# Simulan el catálogo S/4HANA Cloud. En producción → API_SALES_ORDER_SRV

materials = {
    "MAT-1001": {"description": "Bomba centrífuga 5HP",       "price_usd": 1250.00, "stock": 42},
    "MAT-1002": {"description": "Válvula de control 2 pulgadas", "price_usd": 89.50,   "stock": 310},
    "MAT-1003": {"description": "Motor eléctrico trifásico",  "price_usd": 640.00,  "stock": 0},
    "MAT-1004": {"description": "Sensor de presión industrial","price_usd": 215.75,  "stock": 18},
}

customers = {
    "CUST-500": {"name": "Industrias del Norte S.A.",   "credit_limit_usd": 50000.00, "credit_used_usd": 12000.00},
    "CUST-501": {"name": "Logística Andina Ltda.",      "credit_limit_usd": 15000.00, "credit_used_usd": 14200.00},
}

sales_orders = {}
_order_counter = 1000

# ── TOOLS ──────────────────────────────────────────────────

@mcp.tool(
    name="validate_material_number",
    description="Valida si un material existe en el catálogo SAP y retorna disponibilidad de stock.",
)
def validate_material_number(
    material_id: str = Field(description="Código de material SAP, ej: MAT-1001"),
):
    if material_id not in materials:
        raise ValueError(f"Material {material_id} no existe en el catálogo SAP.")
    mat = materials[material_id]
    return {
        "material_id": material_id,
        "description": mat["description"],
        "stock_disponible": mat["stock"],
        "disponible": mat["stock"] > 0,
    }


@mcp.tool(
    name="check_pricing_conditions",
    description="Calcula el precio total de una línea de orden según catálogo SAP.",
)
def check_pricing_conditions(
    material_id: str = Field(description="Código de material SAP"),
    quantity: int = Field(description="Cantidad solicitada"),
):
    if material_id not in materials:
        raise ValueError(f"Material {material_id} no existe.")
    if quantity <= 0:
        raise ValueError("Cantidad debe ser mayor a cero.")
    unit_price = materials[material_id]["price_usd"]
    total = round(unit_price * quantity, 2)
    return {
        "material_id": material_id,
        "quantity": quantity,
        "unit_price_usd": unit_price,
        "total_price_usd": total,
    }


@mcp.tool(
    name="verify_customer_credit_limit",
    description="Verifica si un cliente tiene crédito suficiente para cubrir el valor de una orden.",
)
def verify_customer_credit_limit(
    customer_id: str = Field(description="Código de cliente SAP, ej: CUST-500"),
    order_amount_usd: float = Field(description="Monto total de la orden"),
):
    if customer_id not in customers:
        raise ValueError(f"Cliente {customer_id} no existe.")
    cust = customers[customer_id]
    available = cust["credit_limit_usd"] - cust["credit_used_usd"]
    approved = order_amount_usd <= available
    return {
        "customer_id": customer_id,
        "customer_name": cust["name"],
        "credit_limit_usd": cust["credit_limit_usd"],
        "credit_available_usd": round(available, 2),
        "order_amount_usd": order_amount_usd,
        "approved": approved,
        "reason": "Crédito suficiente" if approved else "Crédito insuficiente — escalar a SAP Build Process Automation",
    }


@mcp.tool(
    name="create_sales_order",
    description="Crea una orden de venta SAP SD una vez validados material, precio y crédito.",
)
def create_sales_order(
    customer_id: str = Field(description="Código de cliente SAP"),
    material_id: str = Field(description="Código de material SAP"),
    quantity: int = Field(description="Cantidad a ordenar"),
):
    global _order_counter
    if customer_id not in customers:
        raise ValueError(f"Cliente {customer_id} no existe.")
    if material_id not in materials:
        raise ValueError(f"Material {material_id} no existe.")
    if materials[material_id]["stock"] < quantity:
        raise ValueError(f"Stock insuficiente. Disponible: {materials[material_id][\"stock\"]}")
    _order_counter += 1
    order_id = f"SO-{_order_counter}"
    total = round(materials[material_id]["price_usd"] * quantity, 2)
    sales_orders[order_id] = {
        "customer_id": customer_id,
        "material_id": material_id,
        "quantity": quantity,
        "total_usd": total,
        "status": "CREATED",
    }
    materials[material_id]["stock"] -= quantity
    return {
        "order_id": order_id,
        "status": "CREATED",
        "customer_id": customer_id,
        "material_id": material_id,
        "quantity": quantity,
        "total_usd": total,
    }


# ── RESOURCES ──────────────────────────────────────────────

@mcp.resource("sap://materials", mime_type="application/json")
def list_materials() -> list:
    return list(materials.keys())


@mcp.resource("sap://orders", mime_type="application/json")
def list_orders() -> list:
    return list(sales_orders.keys())


# ── PROMPT ─────────────────────────────────────────────────

@mcp.prompt(
    name="validate_full_order",
    description="Flujo completo: material → pricing → crédito → creación de orden SAP SD.",
)
def validate_full_order(
    customer_id: str = Field(description="Código de cliente SAP"),
    material_id: str = Field(description="Código de material SAP"),
    quantity: int = Field(description="Cantidad solicitada"),
) -> list[base.Message]:
    prompt = f"""
    Valida y procesa esta orden de venta SAP SD como lo haría un agente Joule con acceso al ERP.

    Datos: customer_id={customer_id} | material_id={material_id} | quantity={quantity}

    Flujo obligatorio en este orden:
    1. validate_material_number → confirmar existencia y stock
    2. check_pricing_conditions → calcular total de la orden
    3. verify_customer_credit_limit → validar crédito con el total calculado
    4. Si todo OK → create_sales_order
       Si algo falla → NO crear la orden. Documentar la excepción como consultor SAP SD.

    Muestra el resultado de cada paso claramente.
    """
    return [base.UserMessage(prompt)]


if __name__ == "__main__":
    mcp.run(transport="stdio")
'''

with open('sap_sd_server.py', 'w') as f:
    f.write(sap_sd_server_code)

print('✅ sap_sd_server.py guardado en disco')

## CELDA 4 — Cliente MCP + Agente Claude

In [ ]:
import asyncio
import json
import nest_asyncio
from contextlib import AsyncExitStack

import anthropic
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

nest_asyncio.apply()  # Necesario para correr async en Colab

# ── AUDIT LOG ──────────────────────────────────────────────
# Cada llamada a una tool queda registrada — esta es la capa de auditoría
audit_log = []

def log_event(event_type, tool_name=None, input_data=None, output_data=None):
    import datetime
    entry = {
        "timestamp": datetime.datetime.now().isoformat(),
        "event": event_type,
        "tool": tool_name,
        "input": input_data,
        "output": output_data,
    }
    audit_log.append(entry)
    return entry


# ── AGENTE SAP SD ──────────────────────────────────────────
async def run_sap_agent(user_query: str, verbose: bool = True):
    """
    Lanza el servidor MCP SAP SD como subproceso,
    conecta el cliente MCP, y ejecuta el agente Claude
    con las tools SAP SD disponibles.
    """

    client_anthropic = anthropic.Anthropic()

    server_params = StdioServerParameters(
        command="python",
        args=["sap_sd_server.py"],
    )

    async with AsyncExitStack() as stack:
        stdio_transport = await stack.enter_async_context(
            stdio_client(server_params)
        )
        stdio, write = stdio_transport
        session = await stack.enter_async_context(
            ClientSession(stdio, write)
        )
        await session.initialize()

        # Obtener tools disponibles del servidor MCP
        tools_result = await session.list_tools()
        tools = [
            {
                "name": t.name,
                "description": t.description,
                "input_schema": t.inputSchema,
            }
            for t in tools_result.tools
        ]

        if verbose:
            print(f"\n{'='*60}")
            print(f"🔌 Servidor MCP conectado")
            print(f"🛠️  Tools disponibles: {[t['name'] for t in tools]}")
            print(f"{'='*60}")
            print(f"\n👤 Usuario: {user_query}\n")

        log_event("SESSION_START", input_data={"query": user_query, "tools": [t["name"] for t in tools]})

        # ── SYSTEM PROMPT del agente ────────────────────────
        system_prompt = """
        Eres un agente SAP SD especializado en validación de órdenes de venta (Order-to-Cash).
        Operas bajo la estrategia Clean Core: toda la lógica corre en la periferia del ERP a través de MCP.
        Cuando valides una orden, sigue siempre este orden: material → pricing → crédito → creación.
        Si alguna validación falla, detén el proceso y documenta la excepción con claridad.
        Responde en español, de forma clara y estructurada.
        """

        messages = [{"role": "user", "content": user_query}]

        # ── BUCLE AGENTICO ──────────────────────────────────
        # El agente corre hasta que no haya más tool_use (stop_reason = "end_turn")
        while True:
            response = client_anthropic.messages.create(
                model="claude-3-5-sonnet-20241022",
                max_tokens=4096,
                system=system_prompt,
                tools=tools,
                messages=messages,
            )

            # Agregar respuesta del asistente al historial
            messages.append({"role": "assistant", "content": response.content})

            if response.stop_reason == "end_turn":
                # El agente terminó — mostrar respuesta final
                for block in response.content:
                    if hasattr(block, "text"):
                        if verbose:
                            print(f"\n🤖 Agente SAP:\n{block.text}")
                        log_event("AGENT_FINAL_RESPONSE", output_data=block.text)
                break

            if response.stop_reason == "tool_use":
                tool_results = []

                for block in response.content:
                    if block.type == "tool_use":
                        tool_name  = block.name
                        tool_input = block.input

                        if verbose:
                            print(f"⚙️  Ejecutando tool: {tool_name}")
                            print(f"   Input: {json.dumps(tool_input, ensure_ascii=False)}")

                        log_event("TOOL_CALL", tool_name=tool_name, input_data=tool_input)

                        try:
                            result = await session.call_tool(tool_name, tool_input)
                            result_text = json.dumps(
                                [c.text for c in result.content if hasattr(c, "text")],
                                ensure_ascii=False
                            )
                            status = "error" if result.isError else "success"

                            if verbose:
                                print(f"   ✅ Resultado: {result_text}\n")

                        except Exception as e:
                            result_text = json.dumps({"error": str(e)})
                            status = "error"
                            if verbose:
                                print(f"   ❌ Error: {e}\n")

                        log_event("TOOL_RESULT", tool_name=tool_name, output_data=result_text)

                        tool_results.append({
                            "type": "tool_result",
                            "tool_use_id": block.id,
                            "content": result_text,
                            "is_error": status == "error",
                        })

                messages.append({"role": "user", "content": tool_results})

    return audit_log


print('✅ Agente SAP SD listo')
print('   Siguiente celda: ejecuta tu primera consulta')

[texto del vínculo](https://)## CELDA 5 — 🚀 Ejecutar el Agente

### Casos de prueba disponibles:

| Escenario | Cliente | Material | Cantidad | Resultado esperado |
|---|---|---|---|---|
| ✅ Happy path | CUST-500 | MAT-1001 | 5 | Orden creada |
| ❌ Sin stock | CUST-500 | MAT-1003 | 1 | Falla: stock = 0 |
| ❌ Crédito insuficiente | CUST-501 | MAT-1001 | 10 | Falla: crédito agotado |
| ❌ Material inválido | CUST-500 | MAT-9999 | 1 | Falla: no existe |


In [ ]:
# ✅ CASO 1 — Happy path: orden válida, debe crearse
query = """
Valida y procesa esta orden de venta SAP:
- Cliente: CUST-500
- Material: MAT-1001
- Cantidad: 5 unidades
"""

audit_log.clear()  # Limpia log de sesiones anteriores
asyncio.run(run_sap_agent(query))

In [ ]:
# ❌ CASO 2 — Crédito insuficiente: CUST-501 tiene solo $800 disponibles
query = """
Valida esta orden de venta SAP:
- Cliente: CUST-501
- Material: MAT-1001
- Cantidad: 10 unidades
"""

audit_log.clear()
asyncio.run(run_sap_agent(query))

In [ ]:
# ❌ CASO 3 — Sin stock: MAT-1003 tiene stock = 0
query = """
Necesito crear una orden para el cliente CUST-500,
material MAT-1003, 2 unidades.
"""

audit_log.clear()
asyncio.run(run_sap_agent(query))

## CELDA 6 — 📋 Ver Audit Log
Cada tool call queda registrada con timestamp, input y output.

In [ ]:
import json
print('\n📋 AUDIT LOG — Trazabilidad completa de la sesión')
print('='*60)
for i, entry in enumerate(audit_log, 1):
    print(f"\n[{i}] {entry['timestamp']}")
    print(f"    Evento : {entry['event']}")
    if entry['tool']:
        print(f"    Tool   : {entry['tool']}")
    if entry['input']:
        print(f"    Input  : {json.dumps(entry['input'], ensure_ascii=False)}")
    if entry['output']:
        out = str(entry['output'])
        print(f"    Output : {out[:200]}{'...' if len(out) > 200 else ''}")

## CELDA 7 — 🔬 Consulta libre (tu propio caso de prueba)

In [ ]:
# Escribe aquí tu propia consulta en lenguaje natural
mi_consulta = """
¿Qué materiales tengo disponibles en el catálogo SAP?
"""

audit_log.clear()
asyncio.run(run_sap_agent(mi_consulta))

---
## 📌 Próximos pasos (Roadmap del proyecto)

| Versión | Feature | Stack adicional |
|---|---|---|
| V1 ✅ | Material · Precio · Crédito · Pedido | FastMCP + Mock data |
| V2 | `check_atp()` — Available-to-Promise con fechas | Lógica de scheduling |
| V3 | `propose_delivery_split()` — Entrega parcial | Supply Chain logic |
| V4 | `trigger_credit_approval()` — Workflow escalación | SAP Build Process Automation API |
| V5 | APIs reales S/4HANA Cloud via Communication Arrangement | httpx + OAuth2 BTP |

---
**Autor:** Fabian Andrés Acevedo R. | Bogotá, Colombia  
**Certificaciones:** C_S4CPB_2602 · C_LCNC_2601  
**LinkedIn:** linkedin.com/in/fabian-acevedo-sap-cloud-data

In [ ]:
import json
from google.colab import userdata
import requests

# 1. Extracción de variables de entorno
try:
    BTP_TOKEN_URL = userdata.get("BTP_TOKEN_URL").strip()
    BTP_API_BASE_URL = userdata.get("BTP_API_BASE_URL").strip()
    BTP_CLIENT_ID = userdata.get("BTP_CLIENT_ID").strip()
    BTP_CLIENT_SECRET = userdata.get("BTP_CLIENT_SECRET").strip()
    print("✅ Todos los secretos de BTP se cargaron correctamente.")
except Exception as e:
    print("❌ Error al leer los secretos. Verifica los nombres en el panel lateral.")
    print(f"Detalle: {e}")

# FASE 1: Autenticación XSUAA
auth_payload = {"grant_type": "client_credentials"}
print("\n🔄 Solicitando token de acceso a SAP XSUAA...")
response_token = requests.post(
    BTP_TOKEN_URL, data=auth_payload, auth=(BTP_CLIENT_ID, BTP_CLIENT_SECRET)
)

if response_token.status_code == 200:
    access_token = response_token.json().get("access_token")
    print("🔓 ¡Autenticación Exitosa! Nuevo token obtenido.")

    # FASE 2: Invocación con endpoint correcto
    url_trigger = f"{BTP_API_BASE_URL}/workflow/rest/v1/workflow-instances"

    headers = {
        "Authorization": f"Bearer {access_token}",
        "Content-Type": "application/json",
    }

    # PAYLOAD RIGUROSO: Llaves en minúscula estricta según el OpenAPI Spec de SAP Build
    payload_contexto = {
        "definitionId": "us10.06a8bac6trial.creditapprovalarchitecture.credit_Approval_Workflow",
        "context": {
            "salesorder": "900001234",
            "customerid": "CUST_BO_01",
            "customername": "Fabián Acevedo Rojas O2C",
            "orderamount": 75000,
            "creditlimit": 100000,
            "creditexposure": 65000
        }
    }

    print("\n--- 🔍 VALIDACIÓN: PAYLOAD ALINEADO AL SCHEMA ---")
    print(json.dumps(payload_contexto, indent=2))
    print("--------------------------------------------------\n")

    print(f"⚡ Disparando flujo en el endpoint: {url_trigger}")
    response_trigger = requests.post(url_trigger, headers=headers, json=payload_contexto)

    if response_trigger.status_code in [200, 201]:
        print("🚀 ¡PROCESO INICIADO EXITOSAMENTE!")
        print(json.dumps(response_trigger.json(), indent=2))
    else:
        print(f"❌ Error al disparar el flujo (HTTP {response_trigger.status_code})")
        print("Respuesta de SAP:", response_trigger.text)
else:
    print(f"❌ Error de autenticación en XSUAA (HTTP {response_token.status_code})")

In [ ]:
import requests
from google.colab import userdata

# 1. Cargar tus secretos desde el entorno seguro de Colab
BTP_TOKEN_URL = userdata.get('BTP_TOKEN_URL').strip()
BTP_API_BASE_URL = userdata.get('BTP_API_BASE_URL').strip()
BTP_CLIENT_ID = userdata.get('BTP_CLIENT_ID').strip()
BTP_CLIENT_SECRET = userdata.get('BTP_CLIENT_SECRET').strip()

# 2. Obtener el token de acceso operativo
auth_payload = {"grant_type": "client_credentials"}
response_token = requests.post(BTP_TOKEN_URL, data=auth_payload, auth=(BTP_CLIENT_ID, BTP_CLIENT_SECRET))
access_token = response_token.json().get("access_token")

# 3. Consultar la lista real de flujos desplegados en tu runtime
url_definitions = f"{BTP_API_BASE_URL}/workflow/rest/v1/workflow-definitions"
headers = {"Authorization": f"Bearer {access_token}"}
response_defs = requests.get(url_definitions, headers=headers)

if response_defs.status_code == 200:
    definiciones = response_defs.json()
    if not definiciones:
        print("⚠️ Conexión exitosa, pero no se encontraron flujos activos. Verifica si el proyecto está activado en el entorno.")
    else:
        print("🔍 ¡CATÁLOGO DE ENCONTRADOS EN TU ENTORNO SAP! 🔍\n")
        for index, definition in enumerate(definiciones, 1):
            print(f"📊 [Flujo #{index}]")
            print(f"   📌 Nombre Comercial: {definition.get('name')}")
            print(f"   🆔 ID TÉCNICO EXACTO (definitionId): {definition.get('id')}")
            print(f"   🚀 Versión en Runtime: {definition.get('version')}")
            print("-" * 60)
else:
    print(f"❌ Error al consultar el catálogo (HTTP {response_defs.status_code})")
    print(response_defs.text)